# 03 model and trust layer

Design decisions: flight-grouped splits (20 train / 20 calibration / 20 test per family, 5 repetitions); `rx_*` columns excluded from the detector; temperature scaling on calibration windows; class-conditional conformal sets at the flight level at alpha 0.10 and 0.20; leave-one-subtype-out; Whelan live logs as external case studies. Result tables are committed under `reports/<run>`.

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
RUN = "features_v3"
subprocess.run(["pip", "install", "-q", "xgboost", "scikit-learn", "scipy", "pandas"], check=True)
out = P.REPORTS / "v4"
proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_model.py"),
                         "--features", str(P.FEATURES / RUN), "--out", str(out), "--only", "e3"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit code:", proc.wait())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyulog"], check=True)
import numpy as np, pathlib
from pyulog import ULog
wdir = P.FEATURES / "sih_flights_v2" / "whelan_live"
for f in sorted(wdir.glob("*.ulg")):
    u = ULog(str(f)); t0 = u.start_timestamp
    g = next(d for d in u.data_list if d.name == "vehicle_gps_position")
    ts = (g.data["timestamp"] - t0) / 1e6
    lat, lon = g.data["lat"] / 1e7, g.data["lon"] / 1e7
    print(f"\n{f.name}")
    print(f"  lat range {lat.min():.5f}..{lat.max():.5f}  lon range {lon.min():.5f}..{lon.max():.5f}")
    print(f"  jump between consecutive fixes, max: {np.max(np.hypot(np.diff(lat)*111320, np.diff(lon)*111320*np.cos(np.radians(lat[:-1])))):.1f} m")
    for k in ("fix_type", "satellites_used", "jamming_indicator", "noise_per_ms", "eph", "s_variance_m_s"):
        v = g.data[k]
        print(f"  {k:18s} min {np.min(v):8.2f}  median {np.median(v):8.2f}  max {np.max(v):8.2f}")
    e = next((d for d in u.data_list if d.name == "estimator_status"), None)
    if e is not None and "gps_check_fail_flags" in e.data:
        v = e.data["gps_check_fail_flags"]
        print(f"  gps_check_fail_flags nonzero fraction: {(v != 0).mean():.2f}")